In [ ]:
# Import necessary libraries
import matplotlib.pyplot as plt
import numpy as np
from AnastrisTNG import TNGsimulation

In [ ]:
# Import the Gal3DAnalyzer class
from gal3d.analyzer import Gal3DAnalyzer

In [ ]:
# Import additional modules from gal3d
from gal3d.point import Particles
from gal3d.field import SphField
from gal3d.shape import Structure3D
from gal3d.optimization.optimizer import Optimizer
from gal3d.visualization.data_model_residual import show_image_model_residual
from gal3d.visualization.model_projector import ModelProjector
from gal3d.characterization.characterizer import Characterizer

In [ ]:
# Load the TNG50 simulation snapshot
path = '/home/yxi/Simulation/TNG50-1/TNG50-1/output'
snap = 99
snapshot = TNGsimulation.Snapshot(path, snap)

In [ ]:
# Load particle data for a specific subhalo ID
ID = 229934  # Example subhalo ID
sub = snapshot.load_particle(ID, order='star')
sub.physical_units()
coor_trans = sub.face_on(alignwith='star', rmax=8)

# Determine the number of rays based on particle count
Num_rays = min(1024, int(len(sub.s) / 100))
Num_rays = max(Num_rays, int(len(sub.s) / 10000))

In [ ]:
# Extract position and mass data
pos = sub.s['pos'].view(np.ndarray)
mass = sub.s['mass'].view(np.ndarray)

# Calculate softening length and outer density value
softening_length = float(sub['eps'].in_units('kpc'))
Volume = (4 * np.pi * softening_length**3)
outer_value = np.mean(mass) / Volume

In [ ]:
# Initialize Particles object with density estimation parameters
particle = Particles(pos=pos, mass=mass, estimator_kwargs={"k_nearest": 32, "r_cut": 10 * softening_length})

In [ ]:
# Build the spherical field with boundary and profile settings
field = SphField(particle, num_ray=Num_rays, ray_method='fibonacci'
    ).build_field_boundary(inner=softening_length / 2, inner_mode='dist', outer=outer_value, outer_mode='value'
    ).build_profile_sample(num_p=500, step_mode='log'
    ).build_profile_interpolator(interpolator_method='LU'
    ).build_isodensity_profile(method='pair', from_rays_func=False, res_b=0.2, res_c=0.1, num_p=500, interpolator_method='LU')

In [ ]:
# Define the Ellipsoid structure
ellipsoid = Structure3D(coordinate='EulerShift', geometry='Ellipsoid', error_func='sums_dev_rscale', error_method='isodensity_dcall')

In [ ]:
# Define the Ellipsoid_S structure
ellipsoid_s = Structure3D(coordinate='EulerShift', geometry='Ellipsoid_S', error_func='sums_dev_rscale', error_method='isodensity_dcall')

In [ ]:
# Initialize the optimizer
optimizer = Optimizer.get_plugin(plugin='OptimizerScipy')(algorithm='Powell')  # OptimizerScipy Powell

In [ ]:
# Create analyzers for both structures
gal_ellipsoid = Gal3DAnalyzer(particle=particle, field=field, structure=ellipsoid, optimizer=optimizer)
gal_ellipsoid_s = Gal3DAnalyzer(particle=particle, field=field, structure=ellipsoid_s, optimizer=optimizer)

In [ ]:
# Define the fitting radius range
r_min = 3 * softening_length
r_max = 100 * softening_length
r_fit = np.geomspace(max(r_min, field.iso_pro_r[0]), min(field.iso_pro_r[-1], r_max), 300)

In [ ]:
# Fit the Ellipsoid structure
res_ellipsoid = gal_ellipsoid.fit(r=r_fit)

In [ ]:
# Fit the Ellipsoid_S structure
res_ellipsoid_s = gal_ellipsoid_s.fit(r=r_fit)

In [ ]:
# Create model projectors for visualization
ellipsoid_model = ModelProjector.get_plugin('ProjectorLineIntegration')(res_ellipsoid)
ellipsoid_s_model = ModelProjector.get_plugin('ProjectorLineIntegration')(res_ellipsoid_s)

In [ ]:
# Characterize the Ellipsoid structure
bar = Characterizer.get_plugin('Bar')
data = {i: res_ellipsoid[i] for i in ['a', 'eps_ab', 'eps_bc', 'ang1', 'ang2', 'ang3']}
data['pa'] = data['ang1']
bar(data).measure()

In [ ]:
# Characterize the Ellipsoid_S structure
bar = Characterizer.get_plugin('Bar')
data = {i: res_ellipsoid_s[i] for i in ['a', 'eps_ab', 'eps_bc', 'ang1', 'ang2', 'ang3']}
dex = np.argsort(data['a'])
for i in ['a', 'eps_ab', 'eps_bc', 'ang1', 'ang2', 'ang3']:
    data[i] = data[i][dex]
data['pa'] = data['ang1']
bar(data).measure()

In [ ]:
# Visualize the model and residuals for the Ellipsoid structure
box_lh_max = (np.max(res_ellipsoid['a']) // 5 + 1) * 5
zoom_lh_max = min(5., box_lh_max / 3)
show_image_model_residual(particle, ellipsoid_model,
                          large_box_x_range=(-box_lh_max, box_lh_max),
                          large_box_y_range=(-box_lh_max, box_lh_max),
                          zoom_x_range=(-zoom_lh_max, zoom_lh_max),
                          zoom_y_range=(-zoom_lh_max, zoom_lh_max),
                          depth_z_range=(-box_lh_max, box_lh_max))

In [ ]:
# Visualize the model and residuals for the Ellipsoid_S structure
box_lh_max = (np.max(res_ellipsoid_s['a']) // 5 + 1) * 5
zoom_lh_max = min(5., box_lh_max / 3)
show_image_model_residual(particle, ellipsoid_s_model,
                          large_box_x_range=(-box_lh_max, box_lh_max),
                          large_box_y_range=(-box_lh_max, box_lh_max),
                          zoom_x_range=(-zoom_lh_max, zoom_lh_max),
                          zoom_y_range=(-zoom_lh_max, zoom_lh_max),
                          depth_z_range=(-box_lh_max, box_lh_max))

In [ ]:
# Import additional visualization utilities
from gal3d.visualization.data_model_residual import show_image, hist_2d

In [ ]:
# Create a detailed visualization of the fitting results
fig3 = plt.figure(figsize=(10, 6), dpi=150)
gs = fig3.add_gridspec(3, 3, height_ratios=[1, 0.6, 0.2], width_ratios=[1, 1, 2], wspace=0.0, hspace=0)

labelfontsize = 13
textfontsize = 14
legendsize = 12

axs_0 = plt.subplot(gs[0:3, 2])

x_range = (-10, 10)
y_range = (-10, 10)
nbins = 200
face, xs, ys = hist_2d(sub.s['x'], sub.s['y'], weights=sub.s['mass'], nbins=nbins, x_range=x_range, y_range=y_range)
show_image(face, axesObj=axs_0, extent=(*x_range, *y_range))

axs_0.set_xlabel('X [kpc]')

axs_00 = plt.subplot(gs[2, 0:2])
axs_1 = plt.subplot(gs[0, 0:2])
axs_2 = plt.subplot(gs[1, 0:2])


axs_1.scatter(res_ellipsoid_s['a'], res_ellipsoid_s['eps_ab'], label='$\epsilon_{ab-3D}$', s=2, c='r')
axs_1.scatter(res_ellipsoid_s['a'], res_ellipsoid_s['eps_bc'], label='$\epsilon_{bc-3D}$', s=2, c='k')

axs_1.axhline(0.25, c='k', linestyle=':')
axs_1.legend(frameon=True)

axs_00.scatter(res_ellipsoid_s['a'], res_ellipsoid_s.res["fun"], c='k', s=3)


r_cut = np.max(res_ellipsoid_s['a'])

axs_00.set_ylabel(r'Fit error', fontsize=labelfontsize)
axs_00.set_ylim(np.min(res_ellipsoid_s.res["fun"]), 0.15)
axs_00.set_yscale('log')

axs_1.set_xlim(0, r_cut)
axs_2.set_xlim(0, r_cut)
axs_00.set_xlim(0, r_cut)
axs_2.scatter(res_ellipsoid_s['a'], res_ellipsoid_s['parameter'], s=4, c='k')


axs_2.set_yscale('log')
axs_1.set_ylabel('Ellipticity', fontsize=labelfontsize)
axs_1.set_ylim(0.01, 0.95)

axs_2.set_ylim(res_ellipsoid_s['parameter'][res_ellipsoid_s['a'] < r_cut][-1] / 2, 2 * res_ellipsoid_s['parameter'][0])
axs_2.set_ylabel(r'$\rho \ [M_{\odot}/kpc^3]$', fontsize=labelfontsize)

axs_1.set_xticklabels([])
axs_2.set_xticklabels([])

axs_00.set_xlabel('R [kpc]', fontsize=labelfontsize)

axs_1.tick_params(axis='both', which='both', direction='in')
axs_2.tick_params(axis='both', which='both', direction='in')
axs_0.tick_params(axis='both', which='both', direction='out')